# Chapter 5: Eigenvalues & Eigenvectors

The eigenvalue equation reveals the **intrinsic geometry** of a linear transformation:

$$Av = \lambda v$$

- $v$ — **eigenvector**: a special direction the matrix can only scale, not rotate
- $\lambda$ — **eigenvalue**: the scaling factor

## Real-World Applications

| Application | Role |
|---|---|
| Google PageRank | Dominant eigenvector of web link matrix |
| PCA | Eigenvectors of covariance matrix = principal components |
| Graph Clustering | Spectral methods using graph Laplacian eigenvalues |
| Quantum Mechanics | Observable eigenvalues = measurement outcomes |
| Stability Analysis | Sign of eigenvalues determines system stability |

## Learning Objectives
- Understand the geometric meaning of eigenvectors
- Compute eigenvalues via the characteristic equation
- Diagonalise matrices and use $A^k = P D^k P^{-1}$
- Apply the spectral theorem for symmetric matrices


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


## Geometric Intuition

For a **generic** vector $v$: applying $A$ changes both direction and magnitude.
For an **eigenvector** $v$: applying $A$ only scales $v$ by $\lambda$ — direction is preserved.

$$Av = \lambda v \implies \text{" }Av\text{ is parallel to }v\text{ "}$$


In [ ]:
A = np.array([[2, 1], [1, 2]], dtype=float)
vals, vecs = np.linalg.eig(A)

# Plot many unit vectors and their images
fig, ax = plt.subplots(figsize=(7, 7))
angles = np.linspace(0, 2*np.pi, 24, endpoint=False)
unit_vecs = np.array([[np.cos(t), np.sin(t)] for t in angles])

for v in unit_vecs:
    Av = A @ v
    ax.annotate('', xy=v,  xytext=[0,0], arrowprops=dict(arrowstyle='->', color='lightsteelblue', lw=1))
    ax.annotate('', xy=Av, xytext=[0,0], arrowprops=dict(arrowstyle='->', color='steelblue', alpha=0.5, lw=1))

# Highlight eigenvectors
for i, (val, vec) in enumerate(zip(vals, vecs.T)):
    v_unit = vec / np.linalg.norm(vec)
    Av = A @ v_unit
    ax.annotate('', xy=v_unit, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='red', lw=3))
    ax.annotate('', xy=Av, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='darkred', lw=3))
    ax.text(*v_unit*1.1, f'$v_{i+1}$\n$\\lambda={val:.1f}$', color='red', fontsize=11, ha='center')

ax.set(xlim=(-3.5,3.5), ylim=(-3.5,3.5), aspect='equal',
       title='Eigenvectors (red): only stretched, not rotated')
ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
plt.tight_layout(); plt.show()


## The Eigenvalue Equation

$$Av = \lambda v \implies (A - \lambda I)v = 0$$

Non-trivial solution exists iff the **characteristic equation** holds:

$$\det(A - \lambda I) = 0$$

For a $2 \times 2$ matrix this gives a quadratic (characteristic polynomial) whose roots are the eigenvalues.


In [ ]:
A = np.array([[2, 1], [1, 2]], dtype=float)

# Sweep lambda and plot det(A - lambda*I)
lam_range = np.linspace(-1, 5, 500)
det_vals = [np.linalg.det(A - lam*np.eye(2)) for lam in lam_range]

true_eigs = np.linalg.eigvals(A)

plt.figure(figsize=(8, 4))
plt.plot(lam_range, det_vals, 'b', lw=2, label=r'$\det(A - \lambda I)$')
plt.axhline(0, color='k', lw=1)
for ev in true_eigs:
    plt.axvline(ev, color='r', ls='--', lw=2, label=f'$\\lambda={ev:.1f}$')
plt.scatter(true_eigs, [0,0], s=100, c='red', zorder=5)
plt.set_xlabel = plt.xlabel(r'$\lambda$')
plt.xlabel(r'$\lambda$'); plt.ylabel(r'$\det(A-\lambda I)$')
plt.title('Characteristic Polynomial — roots are eigenvalues')
plt.legend(); plt.tight_layout(); plt.show()


## Computing with NumPy

`np.linalg.eig(A)` returns `(eigenvalues, eigenvectors)`.
Eigenvectors are **columns** of the returned matrix.

For **symmetric** matrices use `np.linalg.eigh` — guaranteed real eigenvalues and orthogonal eigenvectors, and more numerically stable.


In [ ]:
A = np.array([[4, 2, 0],
              [2, 3, 1],
              [0, 1, 2]], dtype=float)

vals, vecs = np.linalg.eig(A)

print("Eigenvalues:", vals)
print("\nEigenvectors (columns):\n", vecs)

# Verify Av = lambda*v for each pair
print("\nVerification Av = λv:")
for i, (lam, v) in enumerate(zip(vals, vecs.T)):
    err = np.linalg.norm(A @ v - lam * v)
    print(f"  λ_{i+1}={lam:.4f}: |Av - λv| = {err:.2e}")


## Eigenspaces

The **eigenspace** for eigenvalue $\lambda$ is the null space of $(A - \lambda I)$:

$$E_\lambda = \mathcal{N}(A - \lambda I)$$

- **Algebraic multiplicity**: number of times $\lambda$ appears as a root of $\det(A-\lambda I)=0$
- **Geometric multiplicity**: dimension of the eigenspace $E_\lambda$
- Always: geometric multiplicity ≤ algebraic multiplicity


In [ ]:
# Matrix with repeated eigenvalue
A_rep = np.array([[3, 1, 0],
                  [0, 3, 0],
                  [0, 0, 3]], dtype=float)

vals_rep, vecs_rep = np.linalg.eig(A_rep)
print("Eigenvalues:", vals_rep)
print("Eigenvectors:\n", vecs_rep)

# Eigenspace for lambda=3: null space of (A-3I)
from scipy.linalg import null_space
E3 = null_space(A_rep - 3*np.eye(3))
print(f"\nEigenspace dimension (geometric multiplicity): {E3.shape[1]}")
print("Algebraic multiplicity: 3  (triple root)")


## Diagonalisation

If $A$ has $n$ linearly independent eigenvectors, it is **diagonalisable**:

$$A = P D P^{-1}$$

where $D = \text{diag}(\lambda_1, \ldots, \lambda_n)$ and $P = [v_1 \mid \cdots \mid v_n]$.

This decomposition separates the **directions** ($P$) from the **scaling** ($D$).


In [ ]:
A = np.array([[4, 1], [2, 3]], dtype=float)
vals, P = np.linalg.eig(A)
D = np.diag(vals)
P_inv = np.linalg.inv(P)

print("P =\n", P)
print("D =\n", D)
print("P D P^{-1} =\n", P @ D @ P_inv)
print("Reconstructs A?", np.allclose(P @ D @ P_inv, A))

# D^k is trivial: just raise diagonal entries to power k
k = 5
D_k = np.diag(vals**k)
A_k = P @ D_k @ P_inv
print(f"\nA^{k} via diagonalisation:\n", A_k)
print(f"A^{k} via np.linalg.matrix_power:\n", np.linalg.matrix_power(A.astype(int), k))


## Powers of Matrices

$$A^k = P D^k P^{-1}, \quad D^k = \text{diag}(\lambda_1^k, \ldots, \lambda_n^k)$$

Computing $A^{1000}$ is just as fast as $A^1$ via diagonalisation.

**Key application — Markov Chains:** the state distribution after $k$ steps is $\pi^{(k)} = A^k \pi^{(0)}$.
The long-run distribution is the eigenvector for $\lambda = 1$.


In [ ]:
# 3-state Markov chain: Sunny, Cloudy, Rainy
# Transition matrix T[i,j] = P(next=j | current=i)
T = np.array([[0.7, 0.2, 0.1],
              [0.3, 0.5, 0.2],
              [0.2, 0.3, 0.5]])

# Check: rows sum to 1
assert np.allclose(T.sum(axis=1), 1)

# Compute stationary distribution (eigenvector for lambda=1 of T^T)
vals, vecs = np.linalg.eig(T.T)
stat_idx = np.argmax(np.abs(vals - 1) < 1e-8)
stationary = np.real(vecs[:, stat_idx])
stationary /= stationary.sum()
print("Stationary distribution:", stationary)

# Verify convergence
pi = np.array([1.0, 0.0, 0.0])  # start in Sunny
pi_k_vals = []
for k in [1, 5, 10, 50, 100]:
    pi_k = np.linalg.matrix_power(T, k) @ pi
    pi_k_vals.append(pi_k)
    print(f"T^{k} @ pi_0 = {pi_k.round(4)}")

print("\nConverged to stationary?", np.allclose(pi_k_vals[-1], stationary, atol=1e-4))


## Spectral Theorem (Symmetric Matrices)

For real symmetric $A = A^T$:
1. All eigenvalues are **real**
2. Eigenvectors for distinct eigenvalues are **orthogonal**
3. $A$ is **orthogonally diagonalisable**: $A = Q \Lambda Q^T$

**Spectral decomposition** (sum of rank-1 matrices):

$$A = \sum_{i=1}^n \lambda_i\, q_i q_i^T$$

Each $\lambda_i q_i q_i^T$ is a rank-1 "slice" of $A$.


In [ ]:
# Symmetric matrix
S = np.array([[4, 2, 1],
              [2, 5, 3],
              [1, 3, 6]], dtype=float)

vals_s, Q_s = np.linalg.eigh(S)   # use eigh for symmetric matrices
print("Eigenvalues (real):", vals_s)
print("Q^T Q = I?", np.allclose(Q_s.T @ Q_s, np.eye(3)))

# Spectral reconstruction
S_reconstructed = sum(lam * np.outer(q, q) for lam, q in zip(vals_s, Q_s.T))
print("Spectral reconstruction == S?", np.allclose(S_reconstructed, S))


## Positive (Semi)Definite Matrices

$A$ is **positive semidefinite (PSD)** if:
$$x^T A x \geq 0 \quad \forall x \in \mathbb{R}^n \quad \Leftrightarrow \quad \text{all eigenvalues} \geq 0$$

**Covariance matrices are always PSD.** Understanding their eigenstructure IS Principal Component Analysis.


In [ ]:
np.random.seed(42)

# Generate correlated 2D data
mean  = [0, 0]
cov   = [[3, 2], [2, 2]]
data  = np.random.multivariate_normal(mean, cov, 300)
C     = np.cov(data.T)

vals_c, vecs_c = np.linalg.eigh(C)
print("Covariance matrix:\n", C)
print("Eigenvalues (≥0?):", vals_c)

# Plot data + principal axes
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(*data.T, alpha=0.3, s=15)
ax.set_aspect('equal')

for lam, vec in zip(vals_c, vecs_c.T):
    scale = np.sqrt(lam) * 2
    ax.annotate('', xy=scale*vec, xytext=-scale*vec,
                arrowprops=dict(arrowstyle='<->', color='red', lw=3))

ax.set(title='Data + Principal Axes (preview of PCA)',
       xlabel='Feature 1', ylabel='Feature 2')
plt.tight_layout(); plt.show()


## Summary

| Concept | Formula | NumPy |
|---|---|---|
| Eigenvalue equation | $Av = \lambda v$ | `np.linalg.eig(A)` |
| Characteristic equation | $\det(A-\lambda I)=0$ | `np.linalg.eigvals(A)` |
| Diagonalisation | $A = PDP^{-1}$ | from `eig` output |
| Symmetric matrices | $A = Q\Lambda Q^T$ | `np.linalg.eigh(A)` |
| Matrix power | $A^k = PD^kP^{-1}$ | `np.linalg.matrix_power` |

**Next:** Chapter 6 — Matrix Decompositions: LU, QR, Cholesky, and the mighty SVD.
